# CR4 — Training: Facial Emotion Recognition
**Models:** Custom CNN baseline + EfficientNet-B0 (transfer learning)  
**Strategy:** Phase 1 on FERplus → Phase 2 fine-tune on RAF-DB  
**Authors:** Maher Wali & Sarra Majdoub

Results are saved to `MyDrive/facial_emotion_recognition_results/`

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install kagglehub (already available in Colab but pin version)
!pip install kagglehub scikit-learn -q

In [ ]:
# Kaggle authentication
# Option A (recommended): add KAGGLE_USERNAME and KAGGLE_KEY in
#   Colab -> Secrets (key icon in left sidebar) then run this cell.
# Option B: upload your kaggle.json manually.
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
print('Kaggle credentials set.')

In [ ]:
# Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Download Datasets

In [ ]:
import kagglehub

print('Downloading FERplus...')
FERPLUS_ROOT = kagglehub.dataset_download('arnabkumarroy02/ferplus')
print('FERplus:', FERPLUS_ROOT)

print('Downloading RAF-DB...')
RAFDB_ROOT = kagglehub.dataset_download('shuvoalok/raf-db-dataset')
RAFDB_ROOT = RAFDB_ROOT + '/DATASET'
print('RAF-DB:', RAFDB_ROOT)

In [ ]:
# Quick sanity check on dataset structure
import os
print('FERplus splits:', os.listdir(FERPLUS_ROOT))
print('FERplus train classes:', os.listdir(os.path.join(FERPLUS_ROOT, 'train')))
print('RAF-DB splits:', os.listdir(RAFDB_ROOT))
print('RAF-DB train folders:', sorted(os.listdir(os.path.join(RAFDB_ROOT, 'train'))))

## 3. Configuration

In [ ]:
import os

# Paths
DRIVE_RESULTS = '/content/drive/MyDrive/facial_emotion_recognition_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Hyperparameters
IMG_SIZE    = 112
BATCH_SIZE  = 64
NUM_WORKERS = 2
SEED        = 42

# Phase 1 - FERplus
P1_EPOCHS       = 8
P1_LR           = 1e-3
P1_LR_EFFNET    = 1e-4   # lower LR — partial unfreeze keeps ImageNet features intact

# Phase 2 - RAF-DB (fine-tune)
P2_EPOCHS   = 50
P2_LR       = 1e-4
P2_PATIENCE = 8

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
print('Device:', DEVICE)
print('Results will be saved to:', DRIVE_RESULTS)

## 4. Dataset & Transforms

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from PIL import Image

# 7 canonical emotion classes
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
EMO_IDX  = {e: i for i, e in enumerate(EMOTIONS)}

FOLDER_MAP = {
    # FERplus (exclude contempt)
    'angry': 'angry', 'disgust': 'disgust', 'fear': 'fear',
    'happy': 'happy', 'neutral': 'neutral', 'sad':  'sad',
    'suprise': 'surprise',   # typo fix
    'contempt': None,        # excluded
    # RAF-DB numeric folders
    '1': 'surprise', '2': 'fear',    '3': 'disgust',
    '4': 'happy',    '5': 'sad',     '6': 'angry',  '7': 'neutral',
}


class EmotionDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples, self.labels = [], []
        for folder in sorted(os.listdir(root)):
            fpath = os.path.join(root, folder)
            if not os.path.isdir(fpath):
                continue
            emotion = FOLDER_MAP.get(folder.lower())
            if emotion is None:
                continue
            label = EMO_IDX[emotion]
            for fname in os.listdir(fpath):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append(os.path.join(fpath, fname))
                    self.labels.append(label)

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        img = Image.open(self.samples[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

    def class_counts(self):
        counts = [0] * len(EMOTIONS)
        for l in self.labels: counts[l] += 1
        return counts


def make_weighted_sampler(dataset):
    counts = dataset.class_counts()
    w_cls  = [1.0 / max(c, 1) for c in counts]
    return WeightedRandomSampler([w_cls[l] for l in dataset.labels],
                                  len(dataset.labels))


# ImageNet stats
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(10),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

print('Dataset and transforms ready.')

## 5. Models

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
    def forward(self, x): return self.block(x)


class CustomCNN(nn.Module):
    """4-block CNN baseline. Input: 3 x 112 x 112."""
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32),
            ConvBlock(32,  64),
            ConvBlock(64,  128),
            ConvBlock(128, 256),
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.head(self.gap(self.features(x)).flatten(1))


def get_efficientnet(num_classes=7):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_f  = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_f, num_classes),
    )
    return model


def unfreeze_last_stages(model, n=4):
    for p in model.features.parameters(): p.requires_grad = False
    for i in range(len(model.features) - n, len(model.features)):
        for p in model.features[i].parameters(): p.requires_grad = True
    for p in model.classifier.parameters(): p.requires_grad = True


# Quick param count check
cnn = CustomCNN()
eff = get_efficientnet()
print(f'CNN params      : {sum(p.numel() for p in cnn.parameters())/1e6:.2f}M')
print(f'EfficientNet-B0 : {sum(p.numel() for p in eff.parameters())/1e6:.2f}M')

## 6. Training Utilities

In [ ]:
import copy, csv, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report, confusion_matrix


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma           = gamma
        self.weight          = weight
        self.label_smoothing = label_smoothing
    def forward(self, logits, targets):
        ce   = nn.functional.cross_entropy(logits, targets, weight=self.weight,
                                           label_smoothing=self.label_smoothing,
                                           reduction='none')
        loss = ((1 - torch.exp(-ce)) ** self.gamma) * ce
        return loss.mean()


def class_weights(dataset):
    counts = dataset.class_counts()
    total  = sum(counts)
    return torch.tensor([total / (len(EMOTIONS) * max(c, 1))
                         for c in counts], dtype=torch.float32).to(DEVICE)


def mixup_data(imgs, labels, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    return lam * imgs + (1 - lam) * imgs[idx], labels, labels[idx], lam


def train_epoch(model, loader, optimizer, criterion, use_mixup=False):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if use_mixup:
            mixed, labels_a, labels_b, lam = mixup_data(imgs, labels)
            out  = model(mixed)
            loss = lam * criterion(out, labels_a) + (1 - lam) * criterion(out, labels_b)
            correct += (out.argmax(1) == labels).sum().item()
        else:
            out  = model(imgs)
            loss = criterion(out, labels)
            correct += (out.argmax(1) == labels).sum().item()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total      += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    preds_all, labels_all = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out   = model(imgs)
            loss  = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            total   += imgs.size(0)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    f1 = f1_score(labels_all, preds_all, average='macro', zero_division=0)
    return total_loss / total, correct / total, f1


def run_phase(model, loader_tr, loader_val, criterion, optimizer,
              epochs, patience, label, ckpt_path,
              scheduler=None, use_mixup=False):
    print(f'\n  --- {label} ---')
    logs = []
    best_acc, best_state, no_improve = 0.0, None, 0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc         = train_epoch(model, loader_tr, optimizer, criterion, use_mixup)
        vl_loss, vl_acc, vl_f1 = evaluate(model, loader_val, criterion)
        if scheduler: scheduler.step()
        elapsed = time.time() - t0
        logs.append(dict(epoch=ep,
                         train_loss=round(tr_loss,4), val_loss=round(vl_loss,4),
                         train_acc=round(tr_acc,4),   val_acc=round(vl_acc,4),
                         val_f1=round(vl_f1,4)))
        print(f'  Ep {ep:02d}/{epochs} | '
              f'loss {tr_loss:.4f}/{vl_loss:.4f} | '
              f'acc {tr_acc:.3f}/{vl_acc:.3f} | '
              f'f1 {vl_f1:.3f} | {elapsed:.1f}s')
        if vl_acc > best_acc:
            best_acc, best_state, no_improve = vl_acc, copy.deepcopy(model.state_dict()), 0
            torch.save(best_state, ckpt_path)
        else:
            no_improve += 1
            if no_improve >= patience and ep >= 5:
                print(f'  Early stopping at epoch {ep}.')
                break
    model.load_state_dict(best_state)
    return logs


def generate_report(model, loader, out_dir, model_name):
    """Saves confusion_matrix.png and classification_report.csv/.txt."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(EMOTIONS)))
    ax.set_yticks(range(len(EMOTIONS)))
    ax.set_xticklabels(EMOTIONS, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(EMOTIONS, fontsize=9)
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8,
                    color='white' if cm[i, j] > thresh else 'black')
    ax.set_xlabel('Predicted label', fontsize=10)
    ax.set_ylabel('True label', fontsize=10)
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=11, fontweight='bold')
    plt.tight_layout()
    cm_path = os.path.join(out_dir, 'confusion_matrix.png')
    plt.savefig(cm_path, dpi=150)
    plt.show()
    print(f'Saved: {cm_path}')

    # Classification report
    report_dict = classification_report(
        all_labels, all_preds, target_names=EMOTIONS,
        output_dict=True, zero_division=0,
    )
    rows = []
    for emotion in EMOTIONS:
        r = report_dict[emotion]
        rows.append({'class': emotion,
                     'precision': round(r['precision'], 4),
                     'recall':    round(r['recall'],    4),
                     'f1':        round(r['f1-score'],  4),
                     'support':   int(r['support'])})
    for avg in ('macro avg', 'weighted avg'):
        r = report_dict[avg]
        rows.append({'class': avg,
                     'precision': round(r['precision'], 4),
                     'recall':    round(r['recall'],    4),
                     'f1':        round(r['f1-score'],  4),
                     'support':   int(r['support'])})
    save_csv(rows, os.path.join(out_dir, 'classification_report.csv'))

    report_txt = classification_report(
        all_labels, all_preds, target_names=EMOTIONS, zero_division=0,
    )
    print(report_txt)
    with open(os.path.join(out_dir, 'classification_report.txt'), 'w') as f:
        f.write(report_txt)
    print(f'Saved: {os.path.join(out_dir, "classification_report.csv/.txt")}')


def save_curves(logs_p1, logs_p2, out_dir, model_name):
    os.makedirs(out_dir, exist_ok=True)
    for key, ylabel, fname in [('loss', 'Loss', 'loss_curves.png'),
                                ('acc',  'Accuracy', 'accuracy_curves.png')]:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        fig.suptitle(f'{model_name} — {ylabel} curves', fontsize=13, fontweight='bold')
        for ax, logs, title in zip(axes,
                                   [logs_p1, logs_p2],
                                   ['Phase 1 — FERplus', 'Phase 2 — RAF-DB']):
            eps  = [r['epoch']        for r in logs]
            tr_v = [r[f'train_{key}'] for r in logs]
            vl_v = [r[f'val_{key}']   for r in logs]
            ax.plot(eps, tr_v, 'o-',  color='#4C72B0', label='Train')
            ax.plot(eps, vl_v, 's--', color='#DD8452', label='Val')
            ax.set_title(title, fontsize=11)
            ax.set_xlabel('Epoch')
            ax.set_ylabel(ylabel)
            ax.legend()
            ax.grid(alpha=0.3)
            if key == 'acc': ax.set_ylim(0, 1)
        plt.tight_layout()
        path = os.path.join(out_dir, fname)
        plt.savefig(path, dpi=150)
        plt.show()
        print(f'Saved: {path}')


def save_csv(logs, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=logs[0].keys())
        w.writeheader(); w.writerows(logs)


print('Utilities ready.')

## 7. Train — Baseline CNN

In [ ]:
MODEL_NAME = 'baseline_cnn'
OUT_DIR    = os.path.join(DRIVE_RESULTS, MODEL_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

model_cnn = CustomCNN(num_classes=7).to(DEVICE)
n_params  = sum(p.numel() for p in model_cnn.parameters()) / 1e6
print(f'Model: {MODEL_NAME}  |  Params: {n_params:.2f}M  |  Device: {DEVICE}')

# ── Phase 1 : FERplus ────────────────────────────────────────────────────────
ds_tr_p1  = EmotionDataset(os.path.join(FERPLUS_ROOT, 'train'),      train_tf)
ds_vl_p1  = EmotionDataset(os.path.join(FERPLUS_ROOT, 'validation'), val_tf)
print('FERplus train:', len(ds_tr_p1), '| val:', len(ds_vl_p1))
print('Class counts (train):', dict(zip(EMOTIONS, ds_tr_p1.class_counts())))

ld_tr_p1 = DataLoader(ds_tr_p1, BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True)
ld_vl_p1 = DataLoader(ds_vl_p1, BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

cw_p1        = class_weights(ds_tr_p1)
criterion_p1 = nn.CrossEntropyLoss(weight=cw_p1, label_smoothing=0.1)
optim_p1     = torch.optim.AdamW(model_cnn.parameters(), lr=P1_LR, weight_decay=1e-4)

logs_cnn_p1 = run_phase(model_cnn, ld_tr_p1, ld_vl_p1,
                        criterion_p1, optim_p1,
                        P1_EPOCHS, P2_PATIENCE,
                        'Phase 1 — FERplus',
                        os.path.join(OUT_DIR, 'best_phase1.pth'))
save_csv(logs_cnn_p1, os.path.join(OUT_DIR, 'phase1_log.csv'))
print('Phase 1 complete. Best val acc:', max(r['val_acc'] for r in logs_cnn_p1))

In [ ]:
# ── Phase 2 : RAF-DB ─────────────────────────────────────────────────────────
ds_tr_p2 = EmotionDataset(os.path.join(RAFDB_ROOT, 'train'), train_tf)
ds_vl_p2 = EmotionDataset(os.path.join(RAFDB_ROOT, 'test'),  val_tf)
print('RAF-DB train:', len(ds_tr_p2), '| test:', len(ds_vl_p2))
print('Class counts (train):', dict(zip(EMOTIONS, ds_tr_p2.class_counts())))

ld_tr_p2 = DataLoader(ds_tr_p2, BATCH_SIZE,
                      sampler=make_weighted_sampler(ds_tr_p2),
                      num_workers=NUM_WORKERS, pin_memory=True)
ld_vl_p2 = DataLoader(ds_vl_p2, BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

cw_p2        = class_weights(ds_tr_p2)
criterion_p2 = FocalLoss(gamma=2.0, weight=cw_p2, label_smoothing=0.1)
optim_p2     = torch.optim.AdamW(model_cnn.parameters(), lr=P2_LR, weight_decay=1e-4)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(optim_p2, T_max=P2_EPOCHS)

logs_cnn_p2 = run_phase(model_cnn, ld_tr_p2, ld_vl_p2,
                        criterion_p2, optim_p2,
                        P2_EPOCHS, P2_PATIENCE,
                        'Phase 2 — RAF-DB',
                        os.path.join(OUT_DIR, 'best_phase2.pth'),
                        scheduler=scheduler_p2)
save_csv(logs_cnn_p2, os.path.join(OUT_DIR, 'phase2_log.csv'))

# Final test metrics
_, test_acc, test_f1 = evaluate(model_cnn, ld_vl_p2, criterion_p2)
print(f'\nCNN Final — Test Acc: {test_acc:.4f} | Macro F1: {test_f1:.4f}')
with open(os.path.join(OUT_DIR, 'test_results.txt'), 'w') as f:
    f.write(f'Model         : {MODEL_NAME}\n')
    f.write(f'Test Accuracy : {test_acc:.4f}\n')
    f.write(f'Macro F1      : {test_f1:.4f}\n')

generate_report(model_cnn, ld_vl_p2, OUT_DIR, MODEL_NAME)
save_curves(logs_cnn_p1, logs_cnn_p2, OUT_DIR, 'Baseline CNN')
print('CNN training complete. All results saved to Drive.')

## 8. Train — EfficientNet-B0

In [ ]:
MODEL_NAME = 'efficientnet'
OUT_DIR    = os.path.join(DRIVE_RESULTS, MODEL_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

model_eff = get_efficientnet(num_classes=7).to(DEVICE)
n_params  = sum(p.numel() for p in model_eff.parameters()) / 1e6
print(f'Model: {MODEL_NAME}  |  Params: {n_params:.2f}M  |  Device: {DEVICE}')

# ── Phase 1 : FERplus (partial unfreeze — last 3 stages + head) ───────────────
unfreeze_last_stages(model_eff, n=3)
trainable_p1 = sum(p.numel() for p in model_eff.parameters() if p.requires_grad)
print(f'Phase 1 trainable params: {trainable_p1/1e6:.2f}M (last 3 stages + head)')

# Reuse same FERplus loaders from CNN training
cw_p1_eff        = class_weights(ds_tr_p1)
criterion_eff_p1 = nn.CrossEntropyLoss(weight=cw_p1_eff, label_smoothing=0.1)
optim_eff_p1     = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_eff.parameters()),
    lr=P1_LR_EFFNET, weight_decay=1e-4)

logs_eff_p1 = run_phase(model_eff, ld_tr_p1, ld_vl_p1,
                        criterion_eff_p1, optim_eff_p1,
                        P1_EPOCHS, P2_PATIENCE,
                        'Phase 1 — FERplus (partial unfreeze)',
                        os.path.join(OUT_DIR, 'best_phase1.pth'))
save_csv(logs_eff_p1, os.path.join(OUT_DIR, 'phase1_log.csv'))
print('Phase 1 complete. Best val acc:', max(r['val_acc'] for r in logs_eff_p1))

In [ ]:
# ── Phase 2 : RAF-DB (unfreeze last 4 stages) ─────────────────────────────────
unfreeze_last_stages(model_eff, n=4)
trainable_p2 = sum(p.numel() for p in model_eff.parameters() if p.requires_grad)
print(f'Phase 2 trainable params: {trainable_p2/1e6:.2f}M (last 4 stages + head)')

cw_p2_eff        = class_weights(ds_tr_p2)
criterion_eff_p2 = FocalLoss(gamma=2.0, weight=cw_p2_eff, label_smoothing=0.1)
optim_eff_p2     = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_eff.parameters()),
    lr=P2_LR, weight_decay=1e-4)
scheduler_eff_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim_eff_p2, T_max=P2_EPOCHS)

logs_eff_p2 = run_phase(model_eff, ld_tr_p2, ld_vl_p2,
                        criterion_eff_p2, optim_eff_p2,
                        P2_EPOCHS, P2_PATIENCE,
                        'Phase 2 — RAF-DB (fine-tune)',
                        os.path.join(OUT_DIR, 'best_phase2.pth'),
                        scheduler=scheduler_eff_p2)
save_csv(logs_eff_p2, os.path.join(OUT_DIR, 'phase2_log.csv'))

# Final test metrics
_, test_acc, test_f1 = evaluate(model_eff, ld_vl_p2, criterion_eff_p2)
print(f'\nEfficientNet Final — Test Acc: {test_acc:.4f} | Macro F1: {test_f1:.4f}')
with open(os.path.join(OUT_DIR, 'test_results.txt'), 'w') as f:
    f.write(f'Model         : {MODEL_NAME}\n')
    f.write(f'Test Accuracy : {test_acc:.4f}\n')
    f.write(f'Macro F1      : {test_f1:.4f}\n')

generate_report(model_eff, ld_vl_p2, OUT_DIR, MODEL_NAME)
save_curves(logs_eff_p1, logs_eff_p2, OUT_DIR, 'EfficientNet-B0')
print('EfficientNet training complete. All results saved to Drive.')

## 9. Summary

In [ ]:
print('=' * 50)
print(' FINAL RESULTS SUMMARY')
print('=' * 50)

for model_name, p1_logs, p2_logs in [
    ('Baseline CNN',    logs_cnn_p1, logs_cnn_p2),
    ('EfficientNet-B0', logs_eff_p1, logs_eff_p2),
]:
    best_p1 = max(r['val_acc'] for r in p1_logs)
    best_p2 = max(r['val_acc'] for r in p2_logs)
    best_f1 = max(r['val_f1']  for r in p2_logs)
    ep_p1   = len(p1_logs)
    ep_p2   = len(p2_logs)
    print(f'\n{model_name}')
    print(f'  Phase 1 best val acc : {best_p1:.4f}  ({ep_p1} epochs)')
    print(f'  Phase 2 best val acc : {best_p2:.4f}  ({ep_p2} epochs)')
    print(f'  Phase 2 best F1 macro: {best_f1:.4f}')

print(f'\nAll results saved to: {DRIVE_RESULTS}')
print('Download the results/ folder from Drive to your local project.')